# Medical MNIST - Exploratory Data Analysis

This notebook explores the Medical MNIST dataset.

**Dataset:** https://www.kaggle.com/datasets/andrewmvd/medical-mnist

**Classes:**
- AbdomenCT
- BreastMRI
- CXR (Chest X-Ray)
- ChestCT
- Hand
- HeadCT

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

# Add project root to path
sys.path.append(str(Path.cwd().parent))

from config.paths import Paths
from src.data_loader import load_medical_mnist_data, check_data_exists

%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load and Inspect Data

In [ ]:
# Check if data exists
if not check_data_exists(Paths.RAW_DATA_DIR):
    print("Data not found! Please run: python download_data.py")
else:
    # Load data
    image_paths, labels, class_names = load_medical_mnist_data(Paths.RAW_DATA_DIR)
    
    print(f"Total images: {len(image_paths)}")
    print(f"Number of classes: {len(class_names)}")
    print(f"\nClasses: {class_names}")
    
    # Class distribution
    label_counts = Counter(labels)
    print("\nClass distribution:")
    for class_idx, count in sorted(label_counts.items()):
        print(f"  {class_names[class_idx]}: {count} images")

## 2. Visualize Class Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

class_counts = [label_counts[i] for i in range(len(class_names))]
bars = ax.bar(class_names, class_counts, color=sns.color_palette("husl", len(class_names)))

ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Medical MNIST - Class Distribution', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## 3. Visualize Sample Images from Each Class

In [ ]:
def show_class_samples(class_name, n_samples=5):
    """Show sample images from a specific class"""
    class_idx = class_names.index(class_name)
    class_images = [img for img, lbl in zip(image_paths, labels) if lbl == class_idx]
    
    fig, axes = plt.subplots(1, n_samples, figsize=(15, 3))
    fig.suptitle(f'{class_name} - Sample Images', fontsize=14, fontweight='bold')
    
    for i, ax in enumerate(axes):
        if i < len(class_images):
            img = Image.open(class_images[i])
            ax.imshow(img, cmap='gray')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# Show samples from each class
for class_name in class_names:
    show_class_samples(class_name, n_samples=5)

## 4. Image Statistics

In [ ]:
# Check image dimensions and properties
sample_img = Image.open(image_paths[0])
print(f"Sample image size: {sample_img.size}")
print(f"Sample image mode: {sample_img.mode}")
print(f"Sample image format: {sample_img.format}")

# Compute pixel statistics for a sample
n_samples = 1000
sample_indices = np.random.choice(len(image_paths), min(n_samples, len(image_paths)), replace=False)

pixel_means = []
pixel_stds = []

for idx in sample_indices:
    img = np.array(Image.open(image_paths[idx]).convert('L'))
    pixel_means.append(img.mean())
    pixel_stds.append(img.std())

print(f"\nPixel statistics (sample of {len(sample_indices)} images):")
print(f"  Mean pixel value: {np.mean(pixel_means):.2f} ± {np.std(pixel_means):.2f}")
print(f"  Std pixel value: {np.mean(pixel_stds):.2f} ± {np.std(pixel_stds):.2f}")

## 5. Pixel Intensity Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, class_name in enumerate(class_names):
    # Get images for this class
    class_images = [img for img, lbl in zip(image_paths, labels) if lbl == idx]
    
    # Sample 100 images
    sample_imgs = np.random.choice(len(class_images), min(100, len(class_images)), replace=False)
    
    # Collect pixel values
    all_pixels = []
    for i in sample_imgs:
        img = np.array(Image.open(class_images[i]).convert('L')).flatten()
        all_pixels.extend(img)
    
    # Plot histogram
    axes[idx].hist(all_pixels, bins=50, alpha=0.7, color=sns.color_palette("husl", 6)[idx])
    axes[idx].set_title(f'{class_name}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Pixel Intensity')
    axes[idx].set_ylabel('Frequency')

plt.suptitle('Pixel Intensity Distribution by Class', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Grid Visualization

In [ ]:
fig, axes = plt.subplots(len(class_names), 5, figsize=(15, 18))

for class_idx, class_name in enumerate(class_names):
    # Get images for this class
    class_images = [img for img, lbl in zip(image_paths, labels) if lbl == class_idx]
    
    # Show 5 samples
    for i in range(5):
        if i < len(class_images):
            img = Image.open(class_images[i])
            axes[class_idx, i].imshow(img, cmap='gray')
            axes[class_idx, i].axis('off')
            if i == 0:
                axes[class_idx, i].set_ylabel(class_name, fontsize=12, fontweight='bold', rotation=0, labelpad=80, va='center')

plt.suptitle('Medical MNIST - Sample Images by Class', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 7. Summary

### Key Findings:

1. **Dataset Size:** ~58,000 images across 6 medical imaging classes
2. **Image Size:** 64x64 pixels, grayscale
3. **Class Balance:** Relatively balanced dataset
4. **Image Characteristics:**
   - Grayscale medical images
   - Various anatomical regions
   - Different imaging modalities (CT, MRI, X-ray)

### Recommendations for Modeling:

1. **Data Augmentation:** Use rotation, translation, and brightness adjustments
2. **Normalization:** Compute mean/std from training data
3. **Architecture:** CNN works well for 64x64 images, ResNet for transfer learning
4. **Cross-Validation:** Stratified K-fold to maintain class balance
5. **Evaluation:** Multi-class accuracy with per-class metrics